# Monte Carlo Method

This notebook runs the single-site Monte Carlo wavefunction method.

Cleaned-up scope for this project stage:
- one site only
- no hopping `J`
- supports `fock` and `coherent` initial states
- computes variance from `\u27e8n^2\u27e9 - \u27e8n\u27e9^2`

Method behavior:
- Monte Carlo remains a true trajectory-based method even when `interaction_strength = 0`
- this keeps the method honest for thesis comparison, because dissipation-driven jump statistics still matter
- if `interaction_strength != 0`, the same solver path is used with the interaction term included
- this method remains CPU-based

Example filename:
- `monteCarlo_fock_numOfParticles1_gamma1_time5_dt0p001_numOfSamples100_hilbertSize3.csv`


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)


In [ ]:
import matplotlib.pyplot as plt

try:
    import scienceplots  # noqa: F401
    plt.style.use(["science"])
    USING_SCIENCEPLOTS = True
except ImportError:
    USING_SCIENCEPLOTS = False
    print("scienceplots is not installed; using default matplotlib style.")

from bosonic_dissipation.config import DEFAULT_HILBERT_SIZE, resolve_hilbert_size
from bosonic_dissipation.monte_carlo_method import run_monte_carlo_and_save

global_defaults = {
    "initial_state_type": "fock",
    "num_of_particles": 1,
    "gamma": 1.0,
    "time": 5.0,
    "dt": 1e-3,
    "num_of_samples": 100,
    "hilbert_size": DEFAULT_HILBERT_SIZE,
}

monte_carlo_config = {
    "initial_state_type": global_defaults["initial_state_type"],
    "num_of_particles": global_defaults["num_of_particles"],
    "interaction_strength": 0.0,
    "gamma": global_defaults["gamma"],
    "time": global_defaults["time"],
    "dt": global_defaults["dt"],
    "num_of_samples": global_defaults["num_of_samples"],
    "hilbert_size": global_defaults["hilbert_size"],
    "seed": 123,
}

if monte_carlo_config["hilbert_size"] is None:
    monte_carlo_config["hilbert_size"] = resolve_hilbert_size(
        initial_state_type=monte_carlo_config["initial_state_type"],
        num_of_particles=monte_carlo_config["num_of_particles"],
        hilbert_size=monte_carlo_config["hilbert_size"],
    )

print(f"Using scienceplots: {USING_SCIENCEPLOTS}")
monte_carlo_config

## Runtime Block: Monte Carlo

This is the cleaned project version of your stochastic method. The main average is taken over `num_of_samples` trajectories.


In [ ]:
monte_carlo_result, monte_carlo_csv_path = run_monte_carlo_and_save(
    OUTPUT_DIR,
    initial_state_type=monte_carlo_config["initial_state_type"],
    num_of_particles=monte_carlo_config["num_of_particles"],
    interaction_strength=monte_carlo_config["interaction_strength"],
    gamma=monte_carlo_config["gamma"],
    time=monte_carlo_config["time"],
    dt=monte_carlo_config["dt"],
    num_of_samples=monte_carlo_config["num_of_samples"],
    hilbert_size=monte_carlo_config["hilbert_size"],
    seed=monte_carlo_config["seed"],
)

print(f"Backend used: {monte_carlo_result.backend}")
print(f"Runtime (s): {monte_carlo_result.runtime_seconds:.6f}")
print(f"Fast path used: {monte_carlo_result.fast_path_used}")
print(monte_carlo_result.notes)
print(f"Hilbert size: {monte_carlo_result.hilbert_size}")
if monte_carlo_result.coherent_alpha is not None:
    print(f"Coherent alpha: {monte_carlo_result.coherent_alpha}")
print(f"Saved CSV: {monte_carlo_csv_path}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

label_suffix = monte_carlo_result.initial_state_type.capitalize()

axes[0].plot(
    monte_carlo_result.time_values,
    monte_carlo_result.mean_particle_number,
    label=f"Monte Carlo ({label_suffix})",
)
axes[0].set_title("Mean Particle Number")
axes[0].set_xlabel("Time")
axes[0].set_ylabel("Mean Particle Number")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(
    monte_carlo_result.time_values,
    monte_carlo_result.variance,
    label=f"Monte Carlo ({label_suffix})",
    color="tab:orange",
)
axes[1].set_title("Variance")
axes[1].set_xlabel("Time")
axes[1].set_ylabel("Variance")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
monte_carlo_result.time_values[:5], monte_carlo_result.mean_particle_number[:5], monte_carlo_result.variance[:5]